In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
paultimothymooney_chest_xray_pneumonia_path = kagglehub.dataset_download('paultimothymooney/chest-xray-pneumonia')

print('Data source import complete.')


# 시드값 고정 및 GPU 장비 설정

## 시드값 고정

In [ ]:
import torch
import random
import numpy as np
import os

In [ ]:
seed = 50 # 시드값 고정
os.environ['PYTHONHASHSEED'] = str(seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.beenchmark = False
torch.backends.cudnn.enabled = False

## GPU 장비 설정

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

# 데이터 준비

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
data_path = '/kaggle/input/chest-xray-pneumonia/chest_xray/'

In [ ]:
train_path = data_path + 'train/'
valid_path = data_path + 'val/'
test_path = data_path + 'test/'

## 데이터 증강을 위한 이미지 변환기 정의

In [ ]:
from torchvision import transforms

In [ ]:
# 훈련 데이터용 변환기
transform_train = transforms.Compose([
    transforms.Resize((250,250)),                # 이미지 크기 조정
    transforms.CenterCrop(180),                  # 중앙 이미지 확대
    transforms.RandomHorizontalFlip(0.5),        # 좌우 대칭
    transforms.RandomVerticalFlip(0.2),          # 상하 대칭
    transforms.RandomRotation(20),               # 이미지 회전
    transforms.ToTensor(),                       # 텐서 객체로 변환
    transforms.Normalize((0.485, 0.456, 0.406),
                         (0.229, 0.224, 0.225))  # 정규화
])

In [ ]:
# 테스트 데이터용 변환기
transform_test = transforms.Compose([
    transforms.Resize((250,250)),
    transforms.CenterCrop(180),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406),
                         (0.229, 0.224, 0.225))
])

## 데이터셋 및 데이터 로더 생성

In [ ]:
from torchvision.datasets import ImageFolder

In [ ]:
datasets_train = ImageFolder(root=train_path, transform=transform_train)
datasets_valid = ImageFolder(root=valid_path, transform=transform_test)

In [ ]:
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

In [ ]:
g = torch.Generator()
g.manual_seed(0)

In [ ]:
from torch.utils.data import DataLoader

In [ ]:
batch_size = 8

In [ ]:
loader_train = DataLoader(dataset=datasets_train, batch_size=batch_size,
                          shuffle=True, worker_init_fn=seed_worker,
                          generator=g, num_workers=2)
loader_valid = DataLoader(dataset=datasets_valid, batch_size=batch_size,
                          shuffle=True, worker_init_fn=seed_worker,
                          generator=g, num_workers=2)

# 모델 생성

In [ ]:
!pip install efficientnet-pytorch==0.7.1

In [ ]:
from efficientnet_pytorch import EfficientNet

In [ ]:
model = EfficientNet.from_pretrained('efficientnet-b7', num_classes=2)
model = model.to(device)

In [ ]:
print(f'모델 파라미터 개수: ', sum(param.numel() for param in model.parameters()))

# 모델 훈련 및 성능 검증

## 손실 함수와 옵티마이저 설정

In [ ]:
import torch.nn as nn

In [ ]:
criterion = nn.CrossEntropyLoss()

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

## 훈련 함수 작성

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, f1_score
from tqdm.notebook import tqdm

### 1. 뼈대

In [ ]:
# def train(model, loader_train, loader_valid, criterion, optimizer,
#           scheduler=None, epochs=10, save_file='model_state_dict.pth'):
#     for epoch in range(epochs):
#         # ======================= [훈련] =======================
#         # 미니배치 단위로 훈련
#         for images, labels in tqdm(loader_train):
#             # 기울기 초기화
#             # 순전파
#             # 손실값 계산 (훈련 데이터용)
#             # 역전파
#             # 가중치 갱신
#             # 학습률 갱신

#         # ======================= [검증] =======================
#         for images, labels in loader_valid:
#             # 순전파
#             # 손실값 계산 (검증 데이터용)

#         # ============== [최적 모델의 가중치 찾기] ===============
#         # 현 에폭에서의 검증 데이터 손실값이 지금까지 중 가장 작다면
#         # 현 에폭의 모델 가중치(현재까지의 최적 모델 가중치) 저장

#     return torch.load(save_file) # 최적 모델 가중치 반환

### 2. 코드

In [ ]:
def train(model, loader_train, loader_valid, criterion, optimizer,
          scheduler=None, epochs=10, save_file='model_state_dict.pth'):
    valid_loss_min = np.inf # 최소 손실값 초기화 (검증 데이터용)

    for epoch in range(epochs):
        print(f'에폭 [{epoch+1}/{epochs}]\n-------------------------------------')

        # ======================= [훈련] =======================
        model.train()
        epoch_train_loss = 0

        for images, labels in tqdm(loader_train):
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            epoch_train_loss += loss.item()
            loss.backward()
            optimizer.step()

            if scheduler != None:
                scheduler.step()

        print(f'\t훈련 데이터 손실값: {epoch_train_loss/len(loader_train):.4f}')

        # ======================= [검증] =======================
        model.eval()
        epoch_valid_loss = 0
        preds_list = []
        true_list = []

        with torch.no_grad():
            for images, labels in loader_valid:
                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)
                epoch_valid_loss += loss.item()

                preds = torch.max(outputs.cpu(), dim=1)[1].numpy()
                true = labels.cpu().numpy()

                preds_list.extend(preds)
                true_list.extend(true)

            print(f'\t검증 데이터 손실값: {epoch_valid_loss/len(loader_valid):.4f}')

            val_accuracy = accuracy_score(true_list, preds_list)
            val_recall = recall_score(true_list, preds_list)
            val_f1_score = f1_score(true_list, preds_list)
            print(f'\t정확도: {val_accuracy:.4f} / 재현율: {val_recall:.4f} / F1 점수: {val_f1_score:.4f}')

            # ============== [최적 모델의 가중치 찾기] ===============
            # 현 에폭에서의 검증 데이터 손실값이 지금까지 중 가장 작다면
            # 현 에폭의 모델 가중치(현재까지의 최적 모델 가중치) 저장
            if epoch_valid_loss <= valid_loss_min:
                print(f'\t### 검증 데이터 손실값 감소 ({valid_loss_min:.4f} --> {epoch_valid_loss:.4f}). 모델 저장')
                torch.save(model.state_dict(), save_file)
                valid_loss_min = epoch_valid_loss

    return torch.load(save_file) # 저장한 모델 가중치 반환

## 훈련 및 성능 검증

In [ ]:
# 모델 훈련
model_state_dict = train(model=model,
                         loader_train=loader_train,
                         loader_valid=loader_valid,
                         criterion=criterion,
                         optimizer=optimizer)

In [ ]:
# 최적 가중치 불러오기
model.load_state_dict(model_state_dict)

# 예측 및 평가 결과

In [ ]:
dataset_test = ImageFolder(root=test_path, transform=transform_test)
loader_test = DataLoader(dataset=dataset_test, batch_size=batch_size,
                         shuffle=False, worker_init_fn=seed_worker,
                         generator=g, num_workers=2)

## 예측

In [ ]:
def predict(model,loader_test, return_true=False):
    model.eval()
    preds_list = []
    true_list = []

    with torch.no_grad():
            for images, labels in loader_test:
                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)

                preds = torch.max(outputs.cpu(), dim=1)[1].numpy()
                true = labels.cpu().numpy()

                preds_list.extend(preds)
                true_list.extend(true)

            if return_true:
                return true_list, preds_list
            else:
                return preds_list

In [ ]:
true_list, preds_list = predict(model=model,
                                loader_test=loader_test,
                                return_true=True)

## 평가 결과

In [ ]:
print('#'*5, '최종 예측 결과 평가 점수', '#'*5)
print(f'정확도: {accuracy_score(true_list, preds_list):.4f}')
print(f'재현율: {recall_score(true_list, preds_list):.4f}')
print(f'F1 점수: {f1_score(true_list, preds_list):.4f}')

# 성능 개선

## 모델 생성

In [ ]:
# !pip install efficientnet-pytorch==0.7.1 # 설치

In [ ]:
models_list = []

In [ ]:
from efficientnet_pytorch import EfficientNet

In [ ]:
# 모델 생성
efficientnet_b1 = EfficientNet.from_pretrained('efficientnet-b1', num_classes=2)
efficientnet_b2 = EfficientNet.from_pretrained('efficientnet-b2', num_classes=2)
efficientnet_b3 = EfficientNet.from_pretrained('efficientnet-b3', num_classes=2)

In [ ]:
# 장비 할당
efficientnet_b1 = efficientnet_b1.to(device)
efficientnet_b2 = efficientnet_b2.to(device)
efficientnet_b3 = efficientnet_b3.to(device)

In [ ]:
# 리스트에 모델 저장
models_list.append(efficientnet_b1)
models_list.append(efficientnet_b2)
models_list.append(efficientnet_b3)

In [ ]:
for idx, model in enumerate(models_list):
    num_params = sum(param.numel() for param in model.parameters())
    print(f'모델{idx+1} 파라미터 개수: {num_params}')

### 손실 함수, 옵티마이저, 스케즐러 설정

In [ ]:
import torch.nn as nn

In [ ]:
criterion = nn.CrossEntropyLoss()

In [ ]:
optimizer1 = torch.optim.AdamW(models_list[0].parameters(), lr=0.0006, weight_decay=0.001)
optimizer2 = torch.optim.AdamW(models_list[1].parameters(), lr=0.0006, weight_decay=0.001)
optimizer3 = torch.optim.AdamW(models_list[2].parameters(), lr=0.0006, weight_decay=0.001)

In [ ]:
from transformers import get_cosine_schedule_with_warmup

In [ ]:
epochs = 20

In [ ]:
scheduler1 = get_cosine_schedule_with_warmup(optimizer1,
                                             num_warmup_steps=len(loader_train)*3,
                                             num_training_steps=len(loader_train)*epochs)
scheduler2 = get_cosine_schedule_with_warmup(optimizer2,
                                             num_warmup_steps=len(loader_train)*3,
                                             num_training_steps=len(loader_train)*epochs)
scheduler3 = get_cosine_schedule_with_warmup(optimizer3,
                                             num_warmup_steps=len(loader_train)*3,
                                             num_training_steps=len(loader_train)*epochs)

## 모델 훈련 및 성능 검증

In [ ]:
true_list, preds_list1 = predict(model=models_list[0],
                                loader_test=loader_test,
                                return_true=True)

In [ ]:
preds_list2 = predict(model=models_list[1],
                                loader_test=loader_test)

In [ ]:
preds_list3 = predict(model=models_list[2],
                                loader_test=loader_test)

In [ ]:
print('#'*5, '최종 예측 결과 평가 점수', '#'*5)
print(f'정확도: {accuracy_score(true_list, preds_list1):.4f}')
print(f'재현율: {recall_score(true_list, preds_list1):.4f}')
print(f'F1 점수: {f1_score(true_list, preds_list1):.4f}')

In [ ]:
print('#'*5, '최종 예측 결과 평가 점수', '#'*5)
print(f'정확도: {accuracy_score(true_list, preds_list2):.4f}')
print(f'재현율: {recall_score(true_list, preds_list2):.4f}')
print(f'F1 점수: {f1_score(true_list, preds_list2):.4f}')

In [ ]:
print('#'*5, '최종 예측 결과 평가 점수', '#'*5)
print(f'정확도: {accuracy_score(true_list, preds_list3):.4f}')
print(f'재현율: {recall_score(true_list, preds_list3):.4f}')
print(f'F1 점수: {f1_score(true_list, preds_list3):.4f}')

### 앙상블 예측

In [ ]:
ensemble_preds = []

In [ ]:
for i in range(len(preds_list1)):
    pred_element = np.round((preds_list1[i]+preds_list2[i]+preds_list3[i])/3)
    ensemble_preds.append(pred_element)

### 평가 결과

In [ ]:
print('#'*5, '최종 예측 결과 평가 점수', '#'*5)
print(f'정확도: {accuracy_score(true_list, ensemble_preds):.4f}')
print(f'재현율: {recall_score(true_list, ensemble_preds):.4f}')
print(f'F1 점수: {f1_score(true_list, ensemble_preds):.4f}')